# The cell model — full system, one Colab run (Drive-aware)

Everything current, in order, with your Google Drive mounted so the **strong** features load off disk:
1. **Recovery scorecard** — capability axes gated vs known biology.
2. **CompleteCell (Phase 1)** — the full-fidelity, per-gene-queryable cell the ML consumes.
3. **Self-healing LOOP (Phase 2)** — analyse → detect every field → fix/verify (failure-guided, 3 outcomes) → repeat until convergence. Locked ledger + regression check.
4. **Signal-combiner** — one calibrated P(edge) from many signals; trains WITH the Drive dense features (STRING physical, Geneformer embeddings) when present.
5. **Loop again** with the stronger combiner.
6. **DepMap co-essentiality (Phase 3)**.

> Anti-trap throughout: physics > measured > predicted; measured facts are never overwritten.


## 1. Setup — clone the branch + deps


In [ ]:
!git clone --depth 1 -b claude/vectorize-gex-propensity-zp09w8 https://github.com/Nikku03/cell.git 2>/dev/null || (cd cell && git pull)
%cd cell
!pip -q install numpy scipy scikit-learn pandas pyarrow mygene 2>/dev/null   # pyarrow: Tahoe parquet; mygene: ENSG->symbol
import sys, os; sys.path.insert(0, 'colab')
os.makedirs('outputs/orphan', exist_ok=True)
print('ready')


## 2. Mount Drive — restore the cell + wire the STRONG features off disk  **(the key cell)**
Restores the 36 MB `cell_complete.json`, copies the DepMap matrix local, and points the combiner's dense-feature env vars at your Drive files. Paths are tried in a few known locations; adjust if yours differ. Every heavy feature is **optional** — a missing file just turns that feature off, the run still completes.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import glob, gzip, shutil, os
D = '/content/drive/MyDrive'
def first(*pats):
    for p in pats:
        g = sorted(glob.glob(p, recursive=True), key=lambda x: -os.path.getsize(x)) if os.path.sep in p else []
        if g: return g[0]
    return None

# --- 2a. cell_complete.json (git-ignored 36 MB core data) ---
dst = 'outputs/orphan/cell_complete.json'
if not os.path.exists(dst):
    src = first(f'{D}/cell_model/**/cell_complete*.json*', f'{D}/**/cell_complete*.json*')
    assert src, 'cell_complete.json(.gz) not found under MyDrive/cell_model/'
    print('restoring', src)
    (shutil.copyfileobj(gzip.open(src,'rb'), open(dst,'wb')) if src.endswith('.gz') else shutil.copy(src, dst))
import json; print('cell:', len(json.load(open(dst))['genes']), 'genes')

# --- 2b. DepMap gene-effect matrix (co-essentiality) -> copy local for speed ---
os.makedirs('depmap', exist_ok=True)
if not os.path.exists('depmap/CRISPRGeneEffect.csv'):
    ce = first(f'{D}/depmap_data/**/CRISPRGeneEffect.csv', f'{D}/**/CRISPRGeneEffect.csv')
    if ce: print('copying DepMap', ce); shutil.copy(ce, 'depmap/CRISPRGeneEffect.csv')
    else:  print('DepMap not on Drive — Phase 3 cell can download it from figshare instead')
os.environ['DEPMAP_DIR'] = 'depmap'

# --- 2c. STRING physical + Geneformer embeddings -> the combiner auto-uses them ---
sl = first(f'{D}/virtual_cell_data/networks/string_physical*.gz', f'{D}/**/string_physical*.gz')
sa = first(f'{D}/virtual_cell_data/networks/string_aliases*.gz', f'{D}/**/string_aliases*.gz')
gf = first(f'{D}/cell_model/geneformer_gene_emb.npz', f'{D}/**/geneformer_gene_emb.npz')
# dense co-expression matrix — MUST be a BULK expression file (cell-lines x genes, ~0.4 GB). Never the
# largest OmicsExpression*.csv: an all-transcripts / per-profile giant parses to >100 GB and OOMs. Prefer
# the canonical protein-coding TPM matrix, then celltype_expression, then the SMALLEST OmicsExpression<2GB.
def pick_expr():
    for p in (f'{D}/depmap_data/**/OmicsExpressionProteinCodingGenesTPMLogp1.csv',
              f'{D}/**/OmicsExpressionProteinCodingGenesTPMLogp1.csv',
              f'{D}/cell_model/celltype_expression.csv', f'{D}/**/celltype_expression.csv'):
        g = glob.glob(p, recursive=True)
        if g: return g[0]
    cand = [x for x in glob.glob(f'{D}/depmap_data/**/OmicsExpression*.csv', recursive=True)
            if os.path.getsize(x) < 2e9]      # bulk matrices are small; a huge one is the wrong file
    return min(cand, key=os.path.getsize) if cand else None
ex = pick_expr()
# Tahoe-100M: download the FILTERED cell_eval table straight from HF (~900 MB, ~1 min) — never pdex/emb
import fetch_tahoe
if os.path.exists('outputs/orphan/tahoe_vecs.npz'):
    tah = True; print('tahoe: using restored derived cache (no download)')
else:
    tah = fetch_tahoe.fetch()                                  # 14 wide plates from HuggingFace
    if not tah: tah = next(iter(glob.glob(f'{D}/**/tahoe_de', recursive=True)), None)   # Drive fallback
    if isinstance(tah, str): os.environ['TAHOE_DE_DIR'] = tah
esm = first(f'{D}/esm_embeddings.parquet', f'{D}/**/esm_embeddings.parquet')
rx  = first(f'{D}/virtual_cell_data/pathways/reactome_human.txt', f'{D}/**/reactome_human.txt')
sig = first(f'{D}/virtual_cell_data/networks/signor.tsv', f'{D}/**/signor.tsv')
col = first(f'{D}/virtual_cell_data/networks/collectri.tsv', f'{D}/**/collectri.tsv')
if sl: os.environ['STRING_LINKS'] = sl
if sa: os.environ['STRING_ALIASES'] = sa
if gf: os.environ['GENEFORMER_NPZ'] = gf
if ex: os.environ['EXPR_MATRIX'] = ex
if esm: os.environ['ESM_PARQUET'] = esm
if rx: os.environ['REACTOME_TXT'] = rx
if sig: os.environ['SIGNOR_TSV'] = sig
if col: os.environ['COLLECTRI_TSV'] = col
print('STRING:', bool(sl), '| aliases:', bool(sa), '| Geneformer:', bool(gf),
      '| expr-matrix:', bool(ex), '| Tahoe:', bool(tah), '| ESM:', bool(esm),
      '| Reactome:', bool(rx), '| SIGNOR:', bool(sig), '| CollecTRI:', bool(col),
      '| DepMap:', os.path.exists('depmap/CRISPRGeneEffect.csv'))

# --- 2d. restore any saved trained artifacts (so a reconnect doesn't start from scratch) ---
import persist; persist.restore_from_drive(D)


## 3. Recovery scorecard


In [ ]:
!python colab/recovery_scorecard.py


## 4. CompleteCell (Phase 1) — the full-fidelity ML entry point
Every layer reachable per gene at full resolution; `.apply_ledger()` folds in the loop's verified fixes.


In [ ]:
from complete_cell import CompleteCell
cell = CompleteCell()
print('layers:', len(cell.layers()['coverage']), '| genes:', len(cell.genes))
r = cell.gene('TP53'); print('TP53 -> ppi', len(r['ppi_partners']), '| regulates', len(r['regulates']),
      '| complexes', r['complexes'][:2])


## 5. Self-healing LOOP (Phase 2) — analyse → detect → fix/verify, until convergence
First pass (before the combiner is trained it uses the single-lens corroboration). Reports the three outcomes per field, the locked ledger, and the regression check.


In [ ]:
!python colab/phase2_loop.py


## 6. Signal-combiner — WHOLE-CELL: one calibrated model PER relation  *(uses Drive features)*
The combiner isn't PPI-only — it predicts ANY relation by swapping the labels. This trains three: **ppi** (physical binding), **reg** (TF→target regulation), **sig** (signaling). Each keeps the features that actually predict IT (add→measure→keep, per relation) — so watch where the dense datasets land: STRING tends to win PPI, while co-essentiality / co-expression / **Tahoe drug-response** should matter more for **reg** (co-regulation), the relation they're actually about. All Drive features from cell 2c apply.


In [ ]:
# train once, reuse: skip any relation whose model was restored (cell 2d); train the REST in ONE
# process so the big features (DepMap/expr/ESM/STRING) load ONCE, not per-relation (that 3x re-load
# was the ~45-min cost — and it's all CPU, the GPU is idle). FORCE_RETRAIN=1 redoes all (after new data).
import os
need = [rel for rel in ('ppi','reg','sig')
        if os.environ.get('FORCE_RETRAIN') or not os.path.exists(
            'outputs/orphan/signal_combiner.pkl' if rel=='ppi' else f'outputs/orphan/signal_combiner_{rel}.pkl')]
if need:
    print('training (one feature load):', need)
    os.system('python colab/signal_combiner.py ' + ' '.join(need))
else:
    print('all combiners restored from Drive -> skipping retrain (FORCE_RETRAIN=1 to redo)')


## 7. Loop again — now with the trained, stronger combiner
The combiner is picked up as a calibrated lens (with the independence guard). Compare the locked `completion` count to cell 5.


In [ ]:
!python colab/phase2_loop.py


## 7a. Field fixes from Drive — full Reactome pathways + SIGNOR/CollecTRI causal edges
Not combiner features — direct FIELD fixes: full Reactome lifts the pathway-coverage gap (was 23%), and SIGNOR + CollecTRI add directed/signed edges (the causal-direction the model lacked). Each prints the detected format + what it adds; overlays are written, nothing measured is overwritten.


In [ ]:
!python colab/extra_data.py


## 7b. WHOLE-CELL summary — every layer, not just PPI
Genome · all three networks (ppi/reg/sig) with their per-relation combiner AUCs · complexes/SL/LR · kinetics & metabolism · pathways/PTMs · single-cell expression & context · disease/drugs · the ML self-healing outcomes across every field · the capability scorecard.


In [ ]:
!python colab/cell_stats.py


## 8. DepMap co-essentiality (Phase 3) — train the edge model + corroborate the additions
Uses the Drive DepMap matrix from cell 2b (or downloads from figshare if absent).


In [ ]:
import os
if not os.path.exists('depmap/CRISPRGeneEffect.csv'):
    import urllib.request, json
    j = json.loads(urllib.request.urlopen('https://api.figshare.com/v2/articles/25880521').read())
    url = next(f['download_url'] for f in j['files'] if f['name']=='CRISPRGeneEffect.csv')
    print('downloading DepMap (419 MB)…'); urllib.request.urlretrieve(url, 'depmap/CRISPRGeneEffect.csv')
!DEPMAP_DIR=depmap python colab/phase3_depmap.py


## 9. SAVE the trained artifacts to Drive  **(so a disconnect doesn't reset anything)**
Colab wipes the VM on disconnect. This copies the trained combiner, the healed-cell ledger, and every result JSON to `MyDrive/cell_model/artifacts/`. Cell 2d restores them next session — reconnect = instant, and expensive derived features (Tahoe/FEBA later) are cached under `caches/`.


In [ ]:
import persist; persist.save_to_drive(D)


## 10. Extras — reasoned variant, whole-cell kcat, disease  *(optional)*


In [ ]:
!python colab/whole_cell_kcat.py            # test all predicted kcats vs physics + in-vivo floor
!python colab/disease_data.py               # blind disease->target vs real Open Targets genes
from reasoned_variant import ReasonedVariant
rv = ReasonedVariant()
r = rv.predict('HBB','P68871',7,'E','V')   # sickle cell — the gain-of-function blind-spot demo
print('sickle:', r['call'], '| blind_spot:', r.get('ml_blind_spot'))
